In [ ]:
# 🧠 Bayesian Gaussian Mixture Model (BGMM)

import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import BayesianGaussianMixture
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay


# 1️⃣ Load feature_df from CSV
feature_df_path = "../data/feature_df.csv"
feature_df = pd.read_csv(feature_df_path, index_col=0)
print(f"✅ Loaded feature_df from {feature_df_path}")
print(f"   Shape: {feature_df.shape}")
print(f"   Features: {list(feature_df.columns)}")
print()


# 2️⃣ Clean and prepare numeric data

X = feature_df[[
    "mean_price_usd",
    "mean_pct_change",
    "volatility",
    "mean_deviation",
    "spike_count"
]].copy()

X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
X = np.clip(X, -1e6, 1e6)
X = np.array(X, dtype=float)

# Standardize
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)


# 3️⃣ Train Bayesian Gaussian Mixture Model

print("🚀 Training BGMM model (n_components=10)...")
bgmm = BayesianGaussianMixture(
    n_components=10,
    covariance_type='full',
    max_iter=500,
    random_state=42
)
bgmm.fit(X_scaled)
print("✅ BGMM training complete!")


# 4️⃣ Compute anomaly scores and define threshold

scores = -bgmm.score_samples(X_scaled)  # higher score = more anomalous

# dynamic threshold (95th percentile)
threshold = np.percentile(scores, 95)
anomaly_flag = (scores > threshold).astype(int)
print(f"[INFO] Threshold (95th percentile): {threshold:.4f}")


# 5️⃣ Evaluate anomaly spread (optional)

plt.figure(figsize=(8, 5))
sns.histplot(scores, bins=50, kde=True, color="skyblue")
plt.axvline(threshold, color="red", linestyle="--", label="95th Percentile Threshold")
plt.title("BGMM Anomaly Score Distribution")
plt.xlabel("Anomaly Score")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()


# 6️⃣ Integrate anomaly results with features

feature_df["risk_score_BGMM"] = (scores - scores.min()) / (scores.max() - scores.min())
feature_df["anomaly_flag"] = anomaly_flag

# Safe price = lower bound based on risk probability
alpha = 0.25
feature_df["safe_price_usd_BGMM"] = feature_df["mean_price_usd"] * (1 - alpha * feature_df["risk_score_BGMM"])


# 7️⃣ Save model, scaler, and report

os.makedirs("../python_models/models/BGMM_model", exist_ok=True)
os.makedirs("../data/report", exist_ok=True)

joblib.dump(bgmm, "../python_models/models/BGMM_model/bgmm_model.pkl")
joblib.dump(scaler, "../python_models/models/BGMM_model/bgmm_scaler.pkl")
joblib.dump(threshold, "../python_models/models/BGMM_model/bgmm_threshold.pkl")
print("💾 Saved model and scaler → ../python_models/models/BGMM_model/")

bgmm_report = feature_df[[
    "mean_price_usd", "volatility", "spike_count",
    "risk_score_BGMM", "safe_price_usd_BGMM"
]].sort_values(by="risk_score_BGMM", ascending=False)

bgmm_report.to_csv("../data/report/bgmm_safe_prices.csv", index=True)
print("💾 Saved → ../data/report/bgmm_safe_prices.csv")


# 8️⃣ Visualize Safe vs Market Prices

plt.figure(figsize=(10,6))
plt.scatter(bgmm_report["mean_price_usd"], bgmm_report["safe_price_usd_BGMM"],
            c=bgmm_report["risk_score_BGMM"], cmap="coolwarm", alpha=0.7)
plt.colorbar(label="Risk Score (BGMM)")
plt.xlabel("Market Price (USD)")
plt.ylabel("Safe Price (USD)")
plt.title("BGMM — Safe vs Market Prices (Unsupervised Anomaly Model)")
plt.grid(True)
plt.tight_layout()
plt.show()
